**Part 0** — Title

# Strategy Template (Backtestlibrary)

Copy this notebook into your **project** and replace the stub strategy logic with your own entry/exit rules.

**Default behavior in this upgraded template:**
- Streams cache paths (does not load full years into RAM)
- **`BACKTEST_YEARS` in Config is authoritative** — the load cell validates required session files exist for those years (it does not silently shrink the year list)
- Runs **2026** only by default (edit the list for your project)
- Runs both data sources: `normal` and `nasdaq`
- Runs both sessions: `Premarket` and `RTH`
- Runs backtests with max concurrency of **5**
- Exports metrics to `.xlsx`

**Wide split-cache columns** (same as `ChronologicalBacktestEngine`): bare minute keys `4:00`, `Vol 4:00`, `High 4:00`, `Open 4:00` — not `Price 4:00` / `Volume 4:00`. The strategy cell includes `_resolve_price_col` / `_time_column_candidates` helpers for custom intraday logic.

**Cache location expected by default:** `C:\Users\johnn\backtestdata\cache\data` with split files in `pm/` and `rth/`.

**Part 1** — Imports

## 1. Imports & Setup

In [ ]:
import os
import sys
import warnings
import pickle
from datetime import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

_cwd = os.getcwd()
_project_root = _cwd if os.path.isdir(os.path.join(_cwd, "alldata")) else os.path.dirname(_cwd)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from backtestlibrary import (
    BacktestConfig,
    ChronologicalBacktestEngine,
    run_monte_carlo,
    build_full_metrics,
)
from backtestdata import load_cleaned_year_data
from backtestlibrary.bt_types import EntryCandidate, ExitSignal, Position

**Part 2** — Config

## 2. Config

Set data source list, target years, session variations, risk defaults, and run concurrency.

Edit **`BACKTEST_YEARS`** for the years you want to run; the load cell only **checks** that PM/RTH (etc.) paths exist — it will **not** override your list. After changing years or strategy code, set **`FORCE_RERUN_BACKTEST = True`** once or delete the matching pickle under `BACKTEST_RESULTS_DIR`.

Template defaults:
- `BACKTEST_YEARS = [2026]`
- `RUN_DATA_SOURCES = ["normal", "nasdaq"]`
- `SESSION_VARIATIONS = Premarket + RTH`
- `MAX_CONCURRENT_BACKTESTS = 5`

In [ ]:
PROJECT_ROOT = next(
    (c for c in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.isdir(os.path.join(c, "alldata"))),
    os.getcwd(),
)

# Split-session cache root (pm/rth folders with normal_* and nasdaq_* files)
CACHE_DIR = r"C:\Users\johnn\backtestdata\cache\data"

DATA_SOURCES = ["normal", "nasdaq"]
RUN_DATA_SOURCES = ["normal", "nasdaq"]  # run BOTH in one pass for comparison
BACKTEST_YEARS = [2026]  # template default per your request
STARTING_ACCOUNTS = [2000]

# Session variations to run (PM + RTH by default)
PREMARKET_START = time(4, 0)
PREMARKET_END = time(9, 30)
RTH_START = time(9, 30)
RTH_END = time(16, 0)
SESSION_VARIATIONS = [
    ("Premarket", PREMARKET_START, PREMARKET_END),
    ("RTH", RTH_START, RTH_END),
]
_REQUIRED_SESSION_KEYS = ["pm", "rth"]
_SESSION_TO_KEY = {"Premarket": "pm", "RTH": "rth", "After Hours": "ah", "AH": "ah"}

# Engine/risk defaults (keep agnostic; tune per strategy)
FIXED_RISK_PER_TRADE = 100
TIMELINE_STEP_SECONDS = 60
MAX_CONCURRENT_BACKTESTS = 5

RUN_BACKTEST = True
RUN_GRID_TEST = False
RUN_MONTE_CARLO = False
MC_RUNS = 5000

BACKTEST_RESULTS_DIR = os.path.join(PROJECT_ROOT, "backtest_results")
os.makedirs(BACKTEST_RESULTS_DIR, exist_ok=True)
# Set True once after changing BACKTEST_YEARS or strategy so cached pickle is not reused.
FORCE_RERUN_BACKTEST = False

**Part 3** — Strategy

## 3. Strategy Class (stub)

Replace `MyStrategy` with your logic: `prepare_day`, `find_entries_for_day`, `check_exit`.

The cell includes **`_time_column_candidates` / `_resolve_price_col` / `_resolve_vol_col` / `_resolve_*_col`** for wide split-cache parquets; **`MyStrategy`** implements a **long-column** example and **returns no entries** on wide data until you subclass.

In [ ]:
# --- Wide intraday helpers (match ChronologicalBacktestEngine + split parquet naming) ---
# Typical columns: bare "4:00", "Vol 4:00", "High 4:00", "Open 4:00" — not "Price 4:00" / "Volume 4:00".
def _time_column_candidates(t):
    return [
        f"{t.hour}:{t.minute:02d}",
        f"{t.hour:02d}:{t.minute:02d}",
        f"{t.hour}:{t.minute:02d}:00",
        f"{t.hour:02d}:{t.minute:02d}:00",
    ]


def _resolve_price_col(t, columns):
    for c in _time_column_candidates(t):
        if c in columns:
            return c
    cand_str = {str(c) for c in _time_column_candidates(t)}
    for col in columns:
        col_s = str(col)
        if col_s in cand_str or any(col_s.endswith(cs) for cs in cand_str):
            return col
    for prefix in ("Price ", "price "):
        for fmt in (f"{prefix}{t.hour}:{t.minute:02d}", f"{prefix}{t.hour:02d}:{t.minute:02d}"):
            if fmt in columns:
                return fmt
    return None


def _resolve_vol_col(t, columns):
    for fmt in (f"Vol {t.hour}:{t.minute:02d}", f"Vol {t.hour:02d}:{t.minute:02d}"):
        if fmt in columns:
            return fmt
    for fmt in (f"Volume {t.hour}:{t.minute:02d}", f"Volume {t.hour:02d}:{t.minute:02d}"):
        if fmt in columns:
            return fmt
    for col in columns:
        s = str(col).strip()
        if s.lower().startswith("vol ") and (
            s.endswith(f"{t.hour}:{t.minute:02d}") or s.endswith(f"{t.hour:02d}:{t.minute:02d}")
        ):
            return col
    return None


def _resolve_high_col(t, columns):
    for prefix in ("High ", "high "):
        for fmt in (f"{prefix}{t.hour}:{t.minute:02d}", f"{prefix}{t.hour:02d}:{t.minute:02d}"):
            if fmt in columns:
                return fmt
    cand_str = f"{t.hour}:{t.minute:02d}"
    for col in columns:
        s = str(col).strip()
        if s.startswith("High ") and (s.endswith(cand_str) or s.endswith(f"{t.hour:02d}:{t.minute:02d}")):
            return col
    return None


def _resolve_open_col(t, columns):
    for prefix in ("Open ", "open "):
        for fmt in (f"{prefix}{t.hour}:{t.minute:02d}", f"{prefix}{t.hour:02d}:{t.minute:02d}"):
            if fmt in columns:
                return fmt
    cand_str = f"{t.hour}:{t.minute:02d}"
    for col in columns:
        s = str(col).strip()
        if s.startswith("Open ") and (s.endswith(cand_str) or s.endswith(f"{t.hour:02d}:{t.minute:02d}")):
            return col
    return None


def _resolve_low_col(t, columns):
    for prefix in ("Low ", "low "):
        for fmt in (f"{prefix}{t.hour}:{t.minute:02d}", f"{prefix}{t.hour:02d}:{t.minute:02d}"):
            if fmt in columns:
                return fmt
    cand_str = f"{t.hour}:{t.minute:02d}"
    for col in columns:
        s = str(col).strip()
        if s.startswith("Low ") and (s.endswith(cand_str) or s.endswith(f"{t.hour:02d}:{t.minute:02d}")):
            return col
    return None


def _is_wide_minute_format(day_df, timeline) -> bool:
    """True if columns look like wide intraday (per-minute keys), not one scalar threshold per row."""
    if day_df.empty or not timeline:
        return False
    return _resolve_price_col(timeline[0], day_df.columns) is not None


class MyStrategy:
    """Agnostic template: supports two data shapes.

    **Long / enriched** (one row per ticker-day with scalar columns): uses `threshold` + `open` + `high`
    from `*_col_candidates`.

    **Wide minute** (split-cache parquet): default `find_entries_for_day` returns **no entries** — subclass
    and use `_resolve_price_col`, `_time_column_candidates`, `timeline`, and `get_price` to implement
    intraday logic (same pattern as `vwapshort.ipynb`).
    """

    def __init__(
        self,
        threshold_col_candidates=None,
        open_col_candidates=None,
        high_col_candidates=None,
        entry_side="short",
    ):
        self.threshold_col_candidates = threshold_col_candidates or [
            "threshold",
            "entry_threshold",
            "trigger_threshold",
            "vwap_threshold",
            "vwap",
        ]
        self.open_col_candidates = open_col_candidates or ["open", "Open", "o"]
        self.high_col_candidates = high_col_candidates or ["high", "High", "h"]
        self.entry_side = str(entry_side).lower().strip()

    @staticmethod
    def _pick_first_existing(columns, candidates):
        for c in candidates:
            if c in columns:
                return c
        return None

    def prepare_day(self, day_df, timeline, get_price):
        if _is_wide_minute_format(day_df, timeline):
            return {"mode": "wide", "timeline": timeline}
        threshold_col = self._pick_first_existing(day_df.columns, self.threshold_col_candidates)
        open_col = self._pick_first_existing(day_df.columns, self.open_col_candidates)
        high_col = self._pick_first_existing(day_df.columns, self.high_col_candidates)
        if threshold_col is None or open_col is None or high_col is None:
            return None

        tick_col = "ticker" if "ticker" in day_df.columns else ("Ticker" if "Ticker" in day_df.columns else None)
        return {
            "mode": "long",
            "threshold_col": threshold_col,
            "open_col": open_col,
            "high_col": high_col,
            "tick_col": tick_col,
        }

    def find_entries_for_day(self, day_df, timeline, get_price, day_context=None):
        ctx = day_context if day_context is not None else self.prepare_day(day_df, timeline, get_price)
        if not ctx:
            return []
        if ctx.get("mode") == "wide":
            return []

        threshold_col = ctx["threshold_col"]
        open_col = ctx["open_col"]
        high_col = ctx["high_col"]
        tick_col = ctx["tick_col"]

        candidates = []
        for row_idx, row in day_df.iterrows():
            thr = row.get(threshold_col)
            opn = row.get(open_col)
            hi = row.get(high_col)
            if pd.isna(thr) or pd.isna(opn) or pd.isna(hi):
                continue

            hit = (float(opn) >= float(thr)) or (float(hi) >= float(thr))
            if not hit:
                continue

            ticker_val = row.get(tick_col) if tick_col else row_idx
            entry_price = float(opn) if (float(opn) >= float(thr) and float(opn) > 0) else float(hi)
            if not np.isfinite(entry_price) or entry_price <= 0:
                continue

            candidates.append(
                EntryCandidate(
                    ticker=str(ticker_val).strip(),
                    row_index=row_idx,
                    entry_time=timeline[0] if timeline else None,
                    entry_price=entry_price,
                    side=self.entry_side,
                    metadata={
                        "threshold_col": threshold_col,
                        "open_col": open_col,
                        "high_col": high_col,
                        "threshold": float(thr),
                        "open": float(opn),
                        "high": float(hi),
                    },
                )
            )

        return candidates

    def check_exit(self, position, row, current_time, get_price, day_context=None):
        return None

**Part 4** — Load data

## 4. Load data

Loads split cache **paths only** (PM/RTH/AH maps per source) using `resolve_split_session_cache_paths`.

It validates that **`BACKTEST_YEARS` from Config** have required session files for each run source (no silent shrinking of the year list). Discovery uses a broad year probe so you can see what exists on disk; the backtest still uses only `BACKTEST_YEARS`.

In [ ]:
# Resolve split-session cache PATHS only (streaming-friendly; no full-year RAM load)
from backtestlibrary.data import resolve_split_session_cache_paths

if not RUN_DATA_SOURCES:
    raise RuntimeError("RUN_DATA_SOURCES is empty.")

cleaned_year_data_by_source = {}
for _src in RUN_DATA_SOURCES:
    if _src not in DATA_SOURCES:
        raise ValueError(f"RUN_DATA_SOURCES contains unknown source {_src!r}. Must be one of {DATA_SOURCES}.")

    # Probe a broad year range so printed keys show what exists on disk; BACKTEST_YEARS is not overwritten.
    _probe_years = list(range(2015, 2031))
    _paths_by_session = resolve_split_session_cache_paths(CACHE_DIR, _probe_years, cache_prefix=_src)

    cleaned_year_data_by_source[_src] = {
        sk: _paths_by_session.get(sk, {}) for sk in ("pm", "rth", "ah")
    }

    # Strict: every year in BACKTEST_YEARS must have required session cache for this source
    for y in BACKTEST_YEARS:
        ys = str(y)
        for sk in _REQUIRED_SESSION_KEYS:
            if ys not in cleaned_year_data_by_source[_src].get(sk, {}):
                raise FileNotFoundError(
                    f"Missing cache for data_source={_src!r} year {y} session {sk!r}. "
                    f"Expected under {CACHE_DIR} (e.g. {CACHE_DIR}/{sk}/{_src}_{sk}_{y}.parquet or flat layout)."
                )

print("Run data sources:", RUN_DATA_SOURCES)
print("Years for backtest (from Config):", BACKTEST_YEARS)
for _src, _d in cleaned_year_data_by_source.items():
    print(
        f"Loaded {_src} (paths only): pm =",
        list(_d["pm"].keys()),
        "rth =",
        list(_d["rth"].keys()),
        "ah =",
        list(_d["ah"].keys()),
    )

# Compatibility handle (first source, PM) for ad-hoc debugging
cleaned_year_data = cleaned_year_data_by_source[RUN_DATA_SOURCES[0]]["pm"]

**Part 5** — Run backtest

## 5. Run backtest (in-notebook)

Runs all `data_source x session` combinations in parallel with `MAX_CONCURRENT_BACKTESTS` workers.

**Wide cache:** default `MyStrategy` returns **no entries** on wide minute parquets (subclass + use helpers). **Long/enriched** rows with scalar `threshold` / `open` / `high` will produce trades.

Results are cached in one combined pickle (`strategy_template_backtest_cache_ALLSOURCES_<years>.pkl`); missing task combinations are merged on rerun when `FORCE_RERUN_BACKTEST = False`.

In [ ]:
# Run all data_source × session combinations (strategy remains agnostic)
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

base_tasks = [
    ((data_source, session_name), data_source, session_name, sess_start, sess_end)
    for data_source in RUN_DATA_SOURCES
    for session_name, sess_start, sess_end in SESSION_VARIATIONS
]


def _run_one_combo(item):
    key, data_source, session_name, sess_start, sess_end = item

    if session_name not in _SESSION_TO_KEY:
        raise ValueError(f"Unknown session_name {session_name!r}.")
    session_key = _SESSION_TO_KEY[session_name]

    src_data = cleaned_year_data_by_source.get(data_source)
    if src_data is None:
        raise RuntimeError(f"No data for source {data_source!r}.")

    task_data_all_years = src_data.get(session_key)
    if not task_data_all_years:
        raise RuntimeError(f"No paths for source={data_source!r} session={session_key!r}.")

    # Strictly enforce BACKTEST_YEARS in case extra years exist in cache folders.
    task_data = {str(y): task_data_all_years[str(y)] for y in BACKTEST_YEARS if str(y) in task_data_all_years}
    missing_years = sorted(set(map(str, BACKTEST_YEARS)) - set(task_data.keys()))
    if missing_years:
        raise FileNotFoundError(
            f"Missing cache for source={data_source!r}, session={session_name!r}, years={missing_years}."
        )

    engine = ChronologicalBacktestEngine(
        BacktestConfig(
            session_start=sess_start,
            session_end=sess_end,
            risk_pct_per_trade=None,
            fixed_risk_per_trade=FIXED_RISK_PER_TRADE,
            timeline_step_seconds=TIMELINE_STEP_SECONDS,
            use_library_columns=True,
            defer_column_phase=True,
        )
    )

    # Agnostic defaults: trigger when Open or High is >= threshold-like column.
    strategy = MyStrategy(
        threshold_col_candidates=["threshold", "entry_threshold", "trigger_threshold", "vwap_threshold", "vwap"],
        open_col_candidates=["open", "Open", "o"],
        high_col_candidates=["high", "High", "h"],
        entry_side="short",
    )
    raw_results, metrics_df, equity_curves = engine.run(
        task_data, strategy, STARTING_ACCOUNTS, show_progress=True
    )

    full_metrics, _ = build_full_metrics(raw_results, STARTING_ACCOUNTS)
    full_metrics = full_metrics.copy()
    full_metrics["data_source"] = data_source
    full_metrics["session_type"] = session_name
    return key, raw_results, full_metrics


years_str = "_".join(map(str, BACKTEST_YEARS))
backtest_cache_path = os.path.join(
    BACKTEST_RESULTS_DIR,
    f"strategy_template_backtest_cache_ALLSOURCES_{years_str}.pkl",
)

if RUN_BACKTEST:
    if not FORCE_RERUN_BACKTEST and os.path.isfile(backtest_cache_path):
        with open(backtest_cache_path, "rb") as f:
            _cache = pickle.load(f)
        combined_full_metrics = _cache["combined_full_metrics"]
        raw_results_by_key = dict(_cache.get("raw_results_by_key", {}))

        missing = {t[0] for t in base_tasks} - set(raw_results_by_key.keys())
        if missing:
            missing_tasks = [t for t in base_tasks if t[0] in missing]
            all_metrics = []
            with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_BACKTESTS) as ex:
                futures = [ex.submit(_run_one_combo, t) for t in missing_tasks]
                for fut in as_completed(futures):
                    key, raw_results, full_metrics = fut.result()
                    raw_results_by_key[key] = raw_results
                    all_metrics.append(full_metrics)
            if all_metrics:
                combined_full_metrics = pd.concat(
                    [combined_full_metrics, pd.concat(all_metrics, ignore_index=True)],
                    ignore_index=True,
                )
                with open(backtest_cache_path, "wb") as f:
                    pickle.dump({"combined_full_metrics": combined_full_metrics, "raw_results_by_key": raw_results_by_key}, f)
    else:
        all_metrics = []
        raw_results_by_key = {}
        with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_BACKTESTS) as ex:
            futures = [ex.submit(_run_one_combo, t) for t in base_tasks]
            for fut in as_completed(futures):
                key, raw_results, full_metrics = fut.result()
                raw_results_by_key[key] = raw_results
                all_metrics.append(full_metrics)
        combined_full_metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
        with open(backtest_cache_path, "wb") as f:
            pickle.dump({"combined_full_metrics": combined_full_metrics, "raw_results_by_key": raw_results_by_key}, f)
else:
    combined_full_metrics = pd.DataFrame()
    raw_results_by_key = {}
    print("Backtest is currently disabled (RUN_BACKTEST=False).")

# Backwards-compatible handles
full_metrics = combined_full_metrics
raw_results = raw_results_by_key.get((RUN_DATA_SOURCES[0], SESSION_VARIATIONS[0][0]), {})
metrics_df = combined_full_metrics
print("Tasks configured:", len(base_tasks) if RUN_BACKTEST else 0)
print("Total trades:", int(combined_full_metrics["total_trades"].sum()) if (not combined_full_metrics.empty and "total_trades" in combined_full_metrics.columns) else 0)

In [ ]:
# Part 5a — Strategy-agnostic grid search (toggle)
if RUN_GRID_TEST:
    print("RUN_GRID_TEST=True, but grid implementation is intentionally paused in this template.")
    # Paste/restore your grid-search implementation here when ready.
else:
    print("Grid test is currently disabled (RUN_GRID_TEST=False).")


**Part 6** — Metrics

## 6. Metrics (formatted)

Displays full metrics and a comparison view with variation identifiers on the left (`data_source`, `session_type`, `year`, `starting_account`).

Exports both full and comparison tables to an `.xlsx` workbook.

In [ ]:
def _fmt_money(x):
    if pd.isna(x):
        return ""
    return f"${float(x):,.2f}"

if combined_full_metrics is None or combined_full_metrics.empty:
    display(combined_full_metrics)
else:
    # Move variation identifiers left for readability
    _var_cols = [c for c in ["data_source", "session_type", "year", "starting_account"] if c in combined_full_metrics.columns]
    _rest = [c for c in combined_full_metrics.columns if c not in _var_cols]
    combined_full_metrics = combined_full_metrics[_var_cols + _rest]

    display_df = combined_full_metrics.copy()
    for col in ["final_balance", "total_pnl", "avg_trade_pnl", "avg_win", "avg_loss"]:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(_fmt_money)
    for col in ["total_return_pct", "win_rate_pct", "stop_pct", "take_profit_pct", "time_exit_pct"]:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{float(x):.2f}%" if pd.notna(x) else "")

    key_cols = [
        "data_source", "session_type", "year", "starting_account",
        "total_trades", "final_balance", "total_return_pct", "win_rate_pct", "total_pnl",
    ]
    key_cols = [c for c in key_cols if c in combined_full_metrics.columns]
    comparison_df = combined_full_metrics[key_cols].sort_values(["total_pnl"], ascending=False).copy()
    for col in ("final_balance", "total_pnl"):
        if col in comparison_df.columns:
            comparison_df[col] = comparison_df[col].apply(_fmt_money)
    if "total_return_pct" in comparison_df.columns:
        comparison_df["total_return_pct"] = comparison_df["total_return_pct"].apply(lambda x: f"{float(x):.2f}%" if pd.notna(x) else "")
    if "win_rate_pct" in comparison_df.columns:
        comparison_df["win_rate_pct"] = comparison_df["win_rate_pct"].apply(lambda x: f"{float(x):.2f}%" if pd.notna(x) else "")

    display(display_df)
    print("Comparison:")
    display(comparison_df)

    _years_str = "_".join(map(str, BACKTEST_YEARS))
    _excel_path = os.path.join(BACKTEST_RESULTS_DIR, f"strategy_template_metrics_ALLSOURCES_{_years_str}.xlsx")
    try:
        with pd.ExcelWriter(_excel_path, engine="openpyxl") as _writer:
            combined_full_metrics.to_excel(_writer, sheet_name="full_metrics", index=False)
            comparison_df.to_excel(_writer, sheet_name="comparison", index=False)
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "Excel export requires 'openpyxl'. Install it with: pip install openpyxl"
        ) from e
    print("Saved Excel:", _excel_path)

**Part 7** — Monte Carlo

## 7. Monte Carlo (formatted)

In [ ]:
if RUN_MONTE_CARLO and raw_results:
    mc_df, _ = run_monte_carlo(raw_results, STARTING_ACCOUNTS, mc_runs=MC_RUNS, seed=42)
    money_cols = [c for c in mc_df.columns if ("balance" in c.lower() or "pnl" in c.lower() or "dollar" in c.lower()) and "pct" not in c.lower()]
    pct_cols = [c for c in mc_df.columns if ("pct" in c.lower() or "percent" in c.lower()) and c not in money_cols]
    for col in money_cols:
        mc_df[col] = mc_df[col].apply(lambda x: _fmt_money(x) if pd.notna(x) else "")
    for col in pct_cols:
        mc_df[col] = mc_df[col].apply(lambda x: f"{float(x):.2f}%" if pd.notna(x) else "")
    display(mc_df)
else:
    print("Monte Carlo skipped (RUN_MONTE_CARLO=False or no results).")

**Using the generic parallel script**

1. Put your strategy in a `.py` module (for example `my_strategy.py`) with `prepare_day`, `find_entries_for_day`, and `check_exit`.
2. From project root, run the library parallel runner with your years/session settings.
3. Use the output pickle for downstream analysis, or mirror the notebook metrics-export pattern to write `.xlsx`.

Example command:
`python -m backtestlibrary.run_strategy_parallel --strategy-module my_strategy --strategy-class MyStrategy --years 2026 --session-start 4:00 --session-end 16:00 --cache-subdir strategy_template`